# 15 · 卷积的直觉：朴素实现与特征图

> **本节属于 Part 6 · 卷积神经网络 (CNN)。**

全连接层把图像当成一长串无结构的像素，丢掉了"邻近像素相关""同一模式可能出现在任意位置"这些图像的关键先验。**卷积**正是为利用这些先验而生。本节我们用最朴素的循环把 2D 卷积写明白，并亲眼看到卷积核如何从图像中"抠"出边缘等局部特征。

## 学习目标

- 理解卷积 = **滑动小窗口做点积**，以及它的两大优势：**局部连接**与**参数共享**
- 用朴素循环实现 `conv2d`（清晰但慢，作为后续高效实现的"金标准"）
- 用手工设计的卷积核在真实图像上**可视化特征图**
- 理解 stride / padding / 多通道

## 直觉与原理

一个卷积核（filter）是一个小矩阵（如 3×3）。把它在图像上**逐位置滑动**，每到一处就和覆盖的图像块做**逐元素相乘再求和**（点积），得到输出的一个像素。整张输出叫**特征图 (feature map)**。

- **局部连接**：每个输出只看输入的一个小邻域 → 抓住局部模式（边缘、纹理）。
- **参数共享**：同一个核滑遍全图 → 参数量极少，且"平移不变"（同样的模式出现在哪都能被检测到）。

输出尺寸： $\text{out} = \lfloor (H + 2\,\text{pad} - k)/\text{stride}\rfloor + 1$。

## 朴素实现（清晰版）

直接按定义写五重循环——慢，但每一步都和定义一一对应，便于理解。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def conv2d_naive(x, w, b, stride=1, pad=0):
    """x:(N,C,H,W)  w:(O,C,kh,kw)  b:(O,)  ->  (N,O,out_h,out_w)"""
    N, C, H, W = x.shape
    O, _, kh, kw = w.shape
    xpad = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    oh = (H + 2 * pad - kh) // stride + 1
    ow = (W + 2 * pad - kw) // stride + 1
    out = np.zeros((N, O, oh, ow))
    for n in range(N):
        for o in range(O):
            for i in range(oh):
                for j in range(ow):
                    region = xpad[n, :, i*stride:i*stride+kh, j*stride:j*stride+kw]
                    out[n, o, i, j] = np.sum(region * w[o]) + b[o]
    return out

print("一个 3x3 核作用在 5x5 输入(pad=1)上，输出仍是 5x5：")
x = np.ones((1, 1, 5, 5)); w = np.ones((1, 1, 3, 3)); b = np.zeros(1)
print(conv2d_naive(x, w, b, pad=1)[0, 0])

## 看卷积"抠"出了什么：边缘检测

用经典的 **Sobel** 算子（一个检测竖直边缘、一个检测水平边缘）作用在一张 MNIST 数字上，看看特征图。

In [ ]:
import minitorch
(Xtr, ytr), _ = minitorch.utils.load_mnist(n_train=10, n_test=1, flatten=False)
img = Xtr[0:1].reshape(1, 1, 28, 28)        # 一张数字

sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
sobel_y = sobel_x.T
w = np.stack([sobel_x, sobel_y])[:, None, :, :]   # (2,1,3,3)
feats = conv2d_naive(img, w, np.zeros(2), pad=1)

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
ax[0].imshow(img[0, 0], cmap="gray"); ax[0].set_title(f"input (digit {ytr[0]})")
ax[1].imshow(np.abs(feats[0, 0]), cmap="gray"); ax[1].set_title("vertical edges")
ax[2].imshow(np.abs(feats[0, 1]), cmap="gray"); ax[2].set_title("horizontal edges")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

竖直 Sobel 核高亮了数字的竖直笔画，水平核高亮了水平笔画——这正是 CNN 第一层在学的东西，只不过 CNN 的卷积核是**自动学出来的**，而非手工设计。

## stride 与 padding

- **stride（步长）**：核每次滑动的距离。stride=2 会让输出尺寸减半（下采样）。
- **padding（填充）**：在输入边缘补 0，常用来让输出和输入同尺寸（如 3×3 核配 pad=1）。

In [ ]:
x = np.random.randn(1, 1, 8, 8); w = np.random.randn(1, 1, 3, 3); b = np.zeros(1)
for s, p in [(1, 0), (1, 1), (2, 0)]:
    out = conv2d_naive(x, w, b, stride=s, pad=p)
    print(f"stride={s}, pad={p}  ->  输出尺寸 {out.shape[2:]} ")

## 验证：朴素实现 vs PyTorch

确认我们的朴素卷积和 PyTorch 的 `conv2d` 给出相同结果（前向）。

In [ ]:
import torch
from minitorch import rel_error
x = np.random.randn(2, 3, 9, 9); w = np.random.randn(4, 3, 3, 3); b = np.random.randn(4)
ours = conv2d_naive(x, w, b, stride=1, pad=1)
theirs = torch.nn.functional.conv2d(torch.tensor(x), torch.tensor(w), torch.tensor(b), stride=1, padding=1).numpy()
print("朴素卷积 vs PyTorch 相对误差:", rel_error(ours, theirs))

## 📦 沉淀进 minitorch

朴素实现胜在**清晰**，但五重循环太慢，无法用于训练。下一节我们用 **im2col** 把卷积转成矩阵乘法，既快又能复用 `matmul` 的自动求导——那才是 `minitorch.nn.Conv2d` 的真正实现，并会与本节的朴素版**对拍**确保一致。

## 小练习

1. **手工核**：设计一个 3×3 的"模糊"核（所有元素 1/9）作用在数字上，观察图像变模糊。
2. **多通道**：把一张 RGB 图像（C=3）喂给 `conv2d_naive`，确认它对各通道求和。
3. **输出尺寸**：不运行代码，手算 `H=28, k=5, stride=2, pad=0` 时的输出尺寸，再用代码验证。

## 小结 & 下一站

✅ 我们用朴素循环实现了卷积，理解了局部连接与参数共享，并亲眼看到卷积核如何抽取边缘特征。

**下一站 → `16_conv_im2col_and_pooling`**：用 **im2col** 把卷积变成矩阵乘法（复用 `matmul` autograd），实现高效的 `Conv2d` 与 `MaxPool2d`，并和本节朴素版对拍。